# 📊 Notebook 2: Exploratory Data Analysis (EDA)
**Project:** Savanna Bank Kenya — Customer Churn Analysis  
**Author:** Okabinini Harriet-Data Analytics Team  
**Date:** 2026-05-08

---

## Objectives
- Understand the churn distribution across regions, branches, and account types
- Compare churners vs active customers across key numeric features
- Identify seasonal/temporal patterns
- Surface the highest-risk customer segments

## 1. Setup & Load Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

CLEAN_PATH = "../.data/cleaned/bank_customer_churn_clean.csv"
VISUALS    = "../outputs/visuals/"

df = pd.read_csv(CLEAN_PATH)
df["account_open_date"]    = pd.to_datetime(df["account_open_date"])
df["last_transaction_date"] = pd.to_datetime(df["last_transaction_date"])

churners = df[df["churn_label"] == 1]
active   = df[df["churn_label"] == 0]

print(f"Dataset: {len(df):,} customers — {len(churners):,} churned ({len(churners)/len(df)*100:.1f}%), {len(active):,} active")

## 2. Churn Distribution Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Overall churn
labels = ["Active", "Churned"]
sizes  = [len(active), len(churners)]
colors = ["#1D9E75", "#E24B4A"]
axes[0].pie(sizes, labels=labels, autopct="%1.1f%%", colors=colors,
            startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2))
axes[0].set_title("Overall Churn Rate", fontweight="bold")

# Churn by region
region_churn = df.groupby("region")["churn_label"].mean().sort_values(ascending=True) * 100
region_churn.plot(kind="barh", ax=axes[1], color="#378ADD", edgecolor="white")
axes[1].set_title("Churn Rate by Region (%)", fontweight="bold")
axes[1].set_xlabel("Churn Rate (%)")
axes[1].axvline(df["churn_label"].mean()*100, color="red", linestyle="--", linewidth=1.5, label="Overall avg")
axes[1].legend(fontsize=9)

# Churn by account type
acct_churn = df.groupby("account_type")["churn_label"].mean().sort_values(ascending=True) * 100
acct_churn.plot(kind="barh", ax=axes[2], color="#534AB7", edgecolor="white")
axes[2].set_title("Churn Rate by Account Type (%)", fontweight="bold")
axes[2].set_xlabel("Churn Rate (%)")
axes[2].axvline(df["churn_label"].mean()*100, color="red", linestyle="--", linewidth=1.5, label="Overall avg")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig(VISUALS + "04_churn_distribution_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Churners vs Active: Numeric Feature Comparison

In [ ]:
compare_cols = [
    "account_balance_ksh", "monthly_income_ksh", "credit_score",
    "num_transactions_30d", "repayment_burden", "num_products_held"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(compare_cols):
    ax = axes[idx]
    ax.hist(active[col].dropna(), bins=40, alpha=0.6, color="#1D9E75",
            label="Active", density=True, edgecolor="white")
    ax.hist(churners[col].dropna(), bins=40, alpha=0.6, color="#E24B4A",
            label="Churned", density=True, edgecolor="white")
    ax.set_title(col.replace("_", " ").title(), fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylabel("Density")

plt.suptitle("Feature Distributions: Churners vs Active Customers", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(VISUALS + "05_churner_vs_active_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Summary Statistics: Churners vs Active

In [ ]:
summary = pd.DataFrame({
    "Active (mean)":   active[compare_cols].mean(),
    "Churned (mean)":  churners[compare_cols].mean(),
    "Diff (%)": ((churners[compare_cols].mean() - active[compare_cols].mean()) / active[compare_cols].mean() * 100)
}).round(2)
summary

## 5. Churn by Employment & Marital Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

emp_churn = df.groupby("employment_status")["churn_label"].mean().sort_values(ascending=False) * 100
emp_churn.plot(kind="bar", ax=axes[0], color="#BA7517", edgecolor="white")
axes[0].set_title("Churn Rate by Employment Status (%)", fontweight="bold")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].tick_params(axis="x", rotation=30)
axes[0].axhline(df["churn_label"].mean()*100, color="red", linestyle="--", label="Overall avg")
axes[0].legend()

mar_churn = df.groupby("marital_status")["churn_label"].mean().sort_values(ascending=False) * 100
mar_churn.plot(kind="bar", ax=axes[1], color="#D4537E", edgecolor="white")
axes[1].set_title("Churn Rate by Marital Status (%)", fontweight="bold")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].tick_params(axis="x", rotation=30)
axes[1].axhline(df["churn_label"].mean()*100, color="red", linestyle="--", label="Overall avg")
axes[1].legend()

plt.tight_layout()
plt.savefig(VISUALS + "06_churn_by_demographics.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Temporal Patterns: Account Opening & Last Activity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Account opening by month
df["open_month"] = df["account_open_date"].dt.to_period("M").astype(str)
monthly_open = df.groupby("open_month")["churn_label"].agg(["count","mean"]).reset_index()
monthly_open = monthly_open.tail(24)
axes[0].bar(range(len(monthly_open)), monthly_open["count"], color="#378ADD", edgecolor="white")
axes[0].set_title("Account Openings — Last 24 Months", fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Accounts Opened")
tick_step = max(1, len(monthly_open)//6)
axes[0].set_xticks(range(0, len(monthly_open), tick_step))
axes[0].set_xticklabels(monthly_open["open_month"].iloc[::tick_step], rotation=45, ha="right")

# days since last transaction distribution
axes[1].hist(active["days_since_last_txn"], bins=40, alpha=0.6, color="#1D9E75",
             label="Active", density=True, edgecolor="white")
axes[1].hist(churners["days_since_last_txn"], bins=40, alpha=0.6, color="#E24B4A",
             label="Churned", density=True, edgecolor="white")
axes[1].set_title("Days Since Last Transaction", fontweight="bold")
axes[1].set_xlabel("Days")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.savefig(VISUALS + "07_temporal_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Correlation Heatmap

In [ ]:
num_cols = ["account_balance_ksh","monthly_income_ksh","credit_score",
            "num_transactions_30d","num_products_held","days_since_last_txn",
            "account_age_months","repayment_burden","loan_to_income_ratio",
            "balance_to_income_ratio","churn_label"]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size": 8})
ax.set_title("Correlation Matrix — Numeric Features", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(VISUALS + "08_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Key EDA Insights

| Finding | Detail |
|---------|--------|
| **Overall churn rate** | 13.4% (668 of 5,000 customers) |
| **Highest-risk region** | Nyeri (15.7%) and Eldoret (15.3%) exceed the average |
| **Income gap** | Churners earn ~52% less per month than active customers |
| **Repayment burden** | Churners carry a mean burden of 0.74 vs 0.41 for active customers — nearly double |
| **Credit score** | Marginally lower for churners (612 vs 619) — not a primary driver alone |
| **Loan type** | Business and Fixed Deposit accounts show slightly elevated churn |

> **Primary churn profile:** Lower-income customers carrying disproportionate loan repayment burdens, concentrated in Nyeri and Eldoret regions.